# 06 - Ingestão Gold: Agregações Analíticas, KPIs e Destino Duplo (Delta & Azure SQL Server)
**Squad 2 — Real Time for Business | Dupla 5 (Grupo 5)**  
**Integrantes:** Zaiden Emiliano Segundo Seleme *(assumindo o escopo do Grupo 5)*  
**Tabelas de Origem (Silver):** `silver/ecommerce_rastreamento_entregas` e `silver/ecommerce_enderecos`  
**Branch:** `feat/squad2-dupla5-rastreamento-enderecos`  

### Diretrizes Arquiteturais da Camada Gold (Sprint 3):
1. **Consumo Exclusivo Silver-to-Gold:** Proibido ler do container `raw` ou da camada `bronze`. O processamento consome 100% de dados limpos e curados das tabelas Delta da camada Silver.
2. **Destino Duplo Obrigatório:**
   * **Delta Lake (ADLS Gen2):** Persistência analítica com otimização (`autoCompact`, `optimizeWrite`).
   * **Azure SQL Server (Data Warehouse):** Replicação no schema oficial `squad2` para visualização executiva (Power BI / Looker).
3. **Data Marts & KPIs Implementados:**
   * **Data Mart 1:** `gold_logistica_pedidos_rota` — Monitoramento de throughput diário de saídas para entrega, volume entregue, falhas e taxa de sucesso.
   * **Data Mart 2:** `gold_performance_transportadoras` — Ranking e scorecard de performance operacional das transportadoras com cálculo de densidade e eficiência.
   * **Data Mart 3:** `gold_distribuicao_geografica_clientes` — Inteligência geográfica de clientes por UF, volume de endereços e endereços padrão.
4. **Rigor Matemático e Resiliência Serverless:** Estratégia de fallback automático (JDBC / pyodbc) para garantir persistência íntegra sem gargalos no cluster.

In [ ]:
# Instalação e verificação de dependências no escopo da sessão Databricks Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyodbc --quiet


In [ ]:
dbutils.library.restartPython()


## 1. Configuração do Ambiente, Credenciais e Conexões FQDN

In [ ]:
import os
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from dotenv import load_dotenv, find_dotenv

# Carregamento resiliente de variáveis de ambiente do .env
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

# Credenciais do ADLS Gen2
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Credenciais do Azure SQL Server
sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

assert client_id and client_secret and sql_password, "Credenciais ADLS/SQL incompletas no .env"

# Configurações OAuth Spark para leitura/escrita FQDN no ADLS Gen2
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# String de conexão JDBC para SQL Server
jdbc_url = f"jdbc:sqlserver://{sql_host}:1433;database={sql_database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

# Caminhos oficiais no ADLS Gen2
base_silver = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo5/silver"
base_gold = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo5/gold"

caminhos = {
    "silver_rastreamento": f"{base_silver}/ecommerce_rastreamento_entregas",
    "silver_enderecos": f"{base_silver}/ecommerce_enderecos",
    "gold_pedidos_rota": f"{base_gold}/gold_logistica_pedidos_rota",
    "gold_performance_transportadoras": f"{base_gold}/gold_performance_transportadoras",
    "gold_distribuicao_geografica": f"{base_gold}/gold_distribuicao_geografica_clientes"
}

print("Ambiente e caminhos da Camada Gold configurados com sucesso!")


## 2. Leitura Segura das Tabelas Curadas da Camada Silver

In [ ]:
print("Lendo tabelas Delta da Camada Silver...")
df_silver_rastreamento = spark.read.format("delta").options(**adls_options).load(caminhos["silver_rastreamento"])
df_silver_enderecos = spark.read.format("delta").options(**adls_options).load(caminhos["silver_enderecos"])

print(f"Silver Rastreamento: {df_silver_rastreamento.count()} registros carregados.")
print(f"Silver Endereços:    {df_silver_enderecos.count()} registros carregados.")


## 3. Construção dos Data Marts Analíticos (KPIs de Negócio)

### 3.1. Data Mart 1: `gold_logistica_pedidos_rota`
Consolida o throughput logístico por dia e status. Permite acompanhar pedidos que saíram para entrega, pedidos entregues com sucesso, falhas operacionais e a taxa de sucesso global.

In [ ]:
df_gold_pedidos_rota = (
    df_silver_rastreamento
    .withColumn("data_evento", F.to_date("dt_evento"))
    .groupBy("data_evento")
    .agg(
        F.count("id_rastreamento").alias("total_eventos"),
        F.countDistinct("id_pedido_ecommerce").alias("total_pedidos_unicos"),
        F.sum(F.when(F.col("status_entrega") == "coletado", 1).otherwise(0)).alias("total_coletados"),
        F.sum(F.when(F.col("status_entrega") == "em_transito", 1).otherwise(0)).alias("total_em_transito"),
        F.sum(F.when(F.col("status_entrega") == "saiu_para_entrega", 1).otherwise(0)).alias("total_em_rota"),
        F.sum(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("total_entregues"),
        F.sum(F.when(F.col("status_entrega").isin("falha_na_entrega", "devolvido"), 1).otherwise(0)).alias("total_falhas")
    )
    .withColumn(
        "taxa_sucesso_pct",
        F.when(
            (F.col("total_entregues") + F.col("total_falhas")) > 0,
            F.round((F.col("total_entregues") / (F.col("total_entregues") + F.col("total_falhas"))) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("data_evento"))
)

print("Data Mart 1 construído:")
df_gold_pedidos_rota.show(5, truncate=False)


### 3.2. Data Mart 2: `gold_performance_transportadoras`
Mede o volume de pacotes movimentados por cada parceiro logístico, taxa de sucesso de entrega e atribui o ranking diário por volume via `dense_rank()`.

In [ ]:
w_rank_transportadora = Window.partitionBy("data_evento").orderBy(F.desc("volume_movimentado"))

df_gold_performance_transportadoras = (
    df_silver_rastreamento
    .withColumn("data_evento", F.to_date("dt_evento"))
    .groupBy("data_evento", "id_transportadora")
    .agg(
        F.count("id_rastreamento").alias("volume_movimentado"),
        F.countDistinct("id_pedido_ecommerce").alias("total_pedidos_atendidos"),
        F.sum(F.when(F.col("status_entrega") == "entregue", 1).otherwise(0)).alias("volume_entregue"),
        F.sum(F.when(F.col("status_entrega").isin("falha_na_entrega", "devolvido"), 1).otherwise(0)).alias("volume_falhas")
    )
    .withColumn(
        "taxa_eficiencia_pct",
        F.when(
            (F.col("volume_entregue") + F.col("volume_falhas")) > 0,
            F.round((F.col("volume_entregue") / (F.col("volume_entregue") + F.col("volume_falhas"))) * 100, 2)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("ranking_volume_dia", F.dense_rank().over(w_rank_transportadora))
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("data_evento"), F.asc("ranking_volume_dia"))
)

print("Data Mart 2 construído:")
df_gold_performance_transportadoras.show(5, truncate=False)


### 3.3. Data Mart 3: `gold_distribuicao_geografica_clientes`
Analisa a penetração geográfica dos clientes por Unidade Federativa (UF), total de endereços cadastrados, clientes únicos e percentual de penetração sobre a base nacional.

In [ ]:
total_clientes_base = df_silver_enderecos.select("id_cliente").distinct().count()

df_gold_distribuicao_geografica = (
    df_silver_enderecos
    .groupBy("estado")
    .agg(
        F.count("id_endereco").alias("total_enderecos"),
        F.countDistinct("id_cliente").alias("total_clientes_unicos"),
        F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).alias("total_enderecos_principais")
    )
    .withColumn(
        "penetracao_clientes_pct",
        F.round((F.col("total_clientes_unicos") / F.lit(total_clientes_base if total_clientes_base > 0 else 1)) * 100, 2)
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("total_clientes_unicos"))
)

print("Data Mart 3 construído:")
df_gold_distribuicao_geografica.show(10, truncate=False)


## 4. Persistência 1: Gravação em Delta Lake no ADLS Gen2

In [ ]:
def salvar_delta_gold(df, caminho_destino, partition_cols=None):
    print(f"Gravando Delta Lake em: {caminho_destino}...")
    writer = (
        df.write.format("delta")
        .options(**adls_options)
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .mode("overwrite")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    
    writer.save(caminho_destino)
    print(f"[SUCESSO] Tabela Delta persistida em {caminho_destino}!")

# Persistindo os 3 Data Marts no Data Lake (Container squad2 / grupo5 / gold)
salvar_delta_gold(df_gold_pedidos_rota, caminhos["gold_pedidos_rota"])
salvar_delta_gold(df_gold_performance_transportadoras, caminhos["gold_performance_transportadoras"])
salvar_delta_gold(df_gold_distribuicao_geografica, caminhos["gold_distribuicao_geografica"])


## 5. Persistência 2: Replicação Analítica Resiliente no Azure SQL Server (`squad2`)

In [ ]:
def salvar_no_sql_resiliente(df, nome_tabela_destino):
    """
    Salva no Azure SQL Server com tentativa via conector Spark SQLServer/JDBC
    e fallback automático via pyodbc fast_executemany para contornar restrições
    de DML no Databricks Serverless.
    """
    print(f"\nIniciando persistência no SQL Server para '{nome_tabela_destino}'...")
    
    # 1. Tentativa nativa com format('sqlserver')
    try:
        print("  -> [Tentativa 1] Gravando via conector nativo 'sqlserver'...")
        df.write \
            .format("sqlserver") \
            .option("host", sql_host) \
            .option("port", "1433") \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .option("database", sql_database) \
            .option("dbtable", nome_tabela_destino) \
            .mode("overwrite") \
            .save()
        print(f"  [SUCESSO] Carga concluída via conector 'sqlserver' na tabela '{nome_tabela_destino}'!")
        return
    except Exception as e_sql:
        print(f"  Aviso: conector 'sqlserver' retornou: {e_sql}")
    
    # 2. Tentativa via JDBC URL padrão
    try:
        print("  -> [Tentativa 2] Tentando via JDBC URL...")
        df.write \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", nome_tabela_destino) \
            .option("user", sql_user) \
            .option("password", sql_password) \
            .mode("overwrite") \
            .save()
        print(f"  [SUCESSO] Carga concluída via JDBC URL na tabela '{nome_tabela_destino}'!")
        return
    except Exception as e_jdbc:
        print(f"  Aviso: tentativa via JDBC retornou: {e_jdbc}")
    
    # 3. Fallback resiliente via pyodbc fast_executemany
    print("  -> [Tentativa 3] Ativando fallback resiliente via pyodbc...")
    import pyodbc
    pdf = df.toPandas()
    conn_str = f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};Encrypt=yes;TrustServerCertificate=no;"
    try:
        conn = pyodbc.connect(conn_str, timeout=30)
    except Exception:
        conn_str_legacy = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};Encrypt=yes;TrustServerCertificate=no;"
        conn = pyodbc.connect(conn_str_legacy, timeout=30)
    
    cursor = conn.cursor()
    cursor.fast_executemany = True
    
    colunas_sql = []
    for c, dtype in pdf.dtypes.items():
        if "int" in str(dtype):
            colunas_sql.append(f"[{c}] BIGINT")
        elif "float" in str(dtype):
            colunas_sql.append(f"[{c}] FLOAT")
        elif "bool" in str(dtype):
            colunas_sql.append(f"[{c}] BIT")
        elif "datetime" in str(dtype):
            colunas_sql.append(f"[{c}] DATETIME2")
        else:
            colunas_sql.append(f"[{c}] NVARCHAR(MAX)")
    
    cursor.execute(f"IF OBJECT_ID('{nome_tabela_destino}', 'U') IS NOT NULL DROP TABLE {nome_tabela_destino};")
    cursor.execute(f"CREATE TABLE {nome_tabela_destino} ({', '.join(colunas_sql)});")
    conn.commit()
    
    placeholders = ", ".join(["?"] * len(pdf.columns))
    insert_sql = f"INSERT INTO {nome_tabela_destino} VALUES ({placeholders})"
    dados_inserir = [tuple(None if pd.isna(v) else v for v in row) for row in pdf.itertuples(index=False)]
    cursor.executemany(insert_sql, dados_inserir)
    conn.commit()
    conn.close()
    print(f"  [SUCESSO] Carga concluída via pyodbc bypass na tabela '{nome_tabela_destino}' ({len(pdf)} linhas)!")

# Carga das três tabelas analíticas no Azure SQL Server
salvar_no_sql_resiliente(df_gold_pedidos_rota, "squad2.gold_logistica_pedidos_rota")
salvar_no_sql_resiliente(df_gold_performance_transportadoras, "squad2.gold_performance_transportadoras")
salvar_no_sql_resiliente(df_gold_distribuicao_geografica, "squad2.gold_distribuicao_geografica_clientes")


## 6. Auditoria de Fechamento e Data Quality da Camada Gold

In [ ]:
print("=== RELATÓRIO EXECUTIVO DE AUDITORIA GOLD ===")
print(f"1. Pedidos em Rota / SLA:            {df_gold_pedidos_rota.count()} registros persistidos.")
print(f"2. Performance por Transportadora:     {df_gold_performance_transportadoras.count()} registros persistidos.")
print(f"3. Distribuição Geográfica de Clientes: {df_gold_distribuicao_geografica.count()} registros persistidos.")
print("=============================================")
print("Pipeline da Camada Gold executado com sucesso e duplo destino 100% sincronizado!")
